<a href="https://colab.research.google.com/github/UTD2026/Mixed_Dataset_Testing_STA/blob/main/Signal_Routing_Frontier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signal-routing cost-quality frontier (q_ce / admission_score vs out_ce)

This notebook is a rewrite of `Dynamic_V2_CE_Routing_With_Compute_HFGenerateFix.ipynb`.
The previous notebook anchored a dynamic CE threshold to the V2 admission fraction and
concluded that dynamic CE routing tied or slightly lost to fixed `CE@0.3` and was **not
cheaper**. This version tests *why* and *what to do instead*.

**Central problem with the old design.** `out_ce` (output cross-entropy) is derived from a
full **base** generation. So an `out_ce`-gated router must run base on 100% of items, then
run adapted on the routed fraction on top:

```text
out_ce routed cost = base_gen(all) + adapted_gen(routed_fraction)   # cost-ADDITIVE
```

It can never beat base-only on cost; it only buys accuracy. The "clean win" the old
notebook wanted (match accuracy at lower cost) is impossible with `out_ce` by construction.

**The idea this notebook tests.** Route on a **pre-generation** signal that is available
before decoding, so you run *exactly one* model per item:

```text
pre-gen routed cost = base_gen(1 - routed_fraction) + adapted_gen(routed_fraction)   # INTERPOLATES
```

`features.jsonl` already stores two pre-generation signals per item, `q_ce` (prompt
cross-entropy) and `admission_score`, both computable from the prompt prefill you already
pay for. If either routes about as well as `out_ce`, you get a router that can be **cheaper
than base-only** whenever adapted generation is cheaper than base (which the medicine
numbers suggest it dramatically is).

**What the notebook produces:**
1. The reference bounds the old table omitted: base-only, adapted-only, **oracle** ceiling, V2, fixed `CE@0.3`.
2. A **cost-quality Pareto frontier** for every candidate signal, with each signal costed under its correct regime.
3. Honest **val/test split**: the operating point is chosen on validation, frozen, and reported on held-out test (no leakage).
4. A **base-vs-adapted token-length diagnostic** to confirm why base generation dominates cost on medicine.
5. A per-dataset **verdict**: is there a pre-gen router that Pareto-dominates base-only / adapted-only, or does one arm just win outright?


## Codebase walk-through

Same repo snapshot as before. What each artifact gives us:

- `cuda_ttl/ab_routing/ab_build_pool.py` writes `pool/features.jsonl` with `route_choice`, `q_ce`, `out_ce`, `admission_score`, `in_final_pool` per `idx`. **All three candidate signals live here.**
- `cuda_ttl/ab_routing/ab_generate.py` writes `preds_base.jsonl` and `preds_adapted.jsonl` (`{idx, question, label, predict}`).
- `cuda_ttl/ab_routing/grading.py` provides `detect_answer_type`, `extract`, `is_correct`; we reuse it to grade both arms.
- `timing_summary.json` (written by the pipeline cell) gives wall-clock seconds per stage, which become per-item generation costs.

The analysis is pure post-hoc: it reads those four files and never regenerates. The heavy
pipeline (pool build, LoRA train, base + adapted generation) is only needed once to create them.


In [ ]:
# ---- repo / pipeline knobs ----
REPO_URL = "https://github.com/UTD2026/rishabh-tlm.git"
WORK = "/content/v2_dynamic_ce"  # change if not on Colab

# Clone fallback knobs (used only if git clone fails).
LOCAL_REPO_ZIP = "/content/rishabh-tlm-ab-routing-2026-07-02(1).zip"
GITHUB_TOKEN_ENV = "GITHUB_TOKEN"  # optional: os.environ["GITHUB_TOKEN"] for private repos
MODEL = "Qwen/Qwen3.5-0.8B"
DATASETS = ["logiqa", "medicine", "geography"]
N_BY_DATASET = {"logiqa": 1000, "medicine": 1000, "geography": 225}

# Set True to (re)build artifacts. GPU-heavy: pool build + LoRA train + base/adapted generation.
# Set False to run ONLY the frontier analysis against artifacts already on disk.
RUN_FULL_PIPELINE = True
SKIP_IF_EXISTS = True

print({
    "repo": REPO_URL, "work": WORK, "model": MODEL,
    "datasets": DATASETS, "run_full_pipeline": RUN_FULL_PIPELINE,
})


## Environment setup

Written for a CUDA GPU box (Colab/H200/Ada). Skip the install cell if your image already has these packages.

In [ ]:
import os, sys, subprocess, json, textwrap, shutil, pathlib, time
from pathlib import Path

Path(WORK).mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("cwd:", os.getcwd())

In [ ]:
# Fresh Colab runtime install / compatibility setup.
# Qwen3.5 needs newer Transformers support. Generation is patched to use Transformers,
# because Colab/vLLM CUDA wheels can mismatch (e.g. libcudart.so.13 missing).
# Runtime > Restart runtime after this cell if pip asks or if imports still show stale versions.
import os, sys, subprocess, shutil

INSTALL_DEPS = True
USE_QWEN35_COMPAT_INSTALL = True  # keep True for MODEL="Qwen/Qwen3.5-0.8B"
INSTALL_VLLM = False  # keep False on Colab unless you have a known-compatible vLLM/CUDA wheel

if INSTALL_DEPS:
    def pip_install(*args):
        cmd = [sys.executable, "-m", "pip", "install", "-U", *args]
        print("RUN:", " ".join(cmd))
        subprocess.check_call(cmd)

    pip_install("pip")
    pip_install("peft", "accelerate", "scipy", "sympy", "ninja", "pandas", "matplotlib", "torchvision", "pillow", "huggingface_hub")

    if USE_QWEN35_COMPAT_INSTALL and "Qwen3.5" in MODEL:
        # HF model card recommends latest Transformers for Qwen3.5.
        pip_install("transformers[serving] @ git+https://github.com/huggingface/transformers.git@main")
    else:
        pip_install("transformers")

    if INSTALL_VLLM:
        # Optional only. The notebook patches ab_generate.py to use Transformers by default.
        # Use this only if your runtime has a vLLM wheel matching its CUDA/PyTorch stack.
        try:
            cmd = [sys.executable, "-m", "pip", "install", "-U", "vllm", "--extra-index-url", "https://wheels.vllm.ai/nightly"]
            print("RUN:", " ".join(cmd))
            subprocess.check_call(cmd)
        except Exception as e:
            print("Nightly vLLM install failed, falling back to normal vLLM:", repr(e))
            pip_install("vllm")

    # Qwen3.5 GDN Triton JIT expects ninja on PATH in this codebase's README.
    ninja = shutil.which("ninja")
    if ninja:
        print("ninja:", ninja)
    else:
        print("WARNING: ninja still not found on PATH")

import torch, transformers
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
if INSTALL_VLLM:
    try:
        import vllm
        print("vllm:", getattr(vllm, "__version__", "unknown"))
    except Exception as e:
        print("vllm import warning:", repr(e))
else:
    print("vllm skipped; ab_generate.py will use Transformers fallback")


In [ ]:
# Clone / update repo, with token + zip fallback
import os, subprocess, shutil, zipfile, glob
from pathlib import Path

repo_dir = Path(WORK) / "rishabh-tlm"


def _run_capture(cmd, cwd=None):
    """Run a shell command and return CompletedProcess while preserving stdout/stderr."""
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )


def _clone_url_with_optional_token(url: str) -> str:
    token = os.environ.get(GITHUB_TOKEN_ENV, "").strip()
    if token and url.startswith("https://github.com/"):
        # Do not print this URL; it contains a secret.
        return url.replace("https://", f"https://{token}@", 1)
    return url


def _unzip_repo(zip_path: Path, dest: Path):
    zip_path = Path(zip_path).expanduser().resolve()
    if not zip_path.exists():
        raise FileNotFoundError(f"Repo zip does not exist: {zip_path}")

    print(f"Using repo zip fallback: {zip_path}")
    tmp = dest.parent / "_repo_zip_extract_tmp"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp)

    # Common cases: zip contains a single top-level folder, or contains repo files directly.
    candidates = [p for p in tmp.iterdir() if p.is_dir()]
    selected = None
    for cand in candidates:
        if (cand / "cuda_ttl" / "ab_routing").exists() or (cand / ".git").exists():
            selected = cand
            break
    if selected is None:
        if (tmp / "cuda_ttl" / "ab_routing").exists():
            selected = tmp
        elif len(candidates) == 1:
            selected = candidates[0]
        else:
            raise RuntimeError(
                "Could not identify repo root inside zip. "
                f"Top-level entries: {[p.name for p in tmp.iterdir()][:20]}"
            )

    if dest.exists():
        shutil.rmtree(dest)
    if selected == tmp:
        shutil.copytree(tmp, dest, dirs_exist_ok=True)
        shutil.rmtree(tmp, ignore_errors=True)
    else:
        shutil.move(str(selected), str(dest))
        shutil.rmtree(tmp, ignore_errors=True)

    if not (dest / "cuda_ttl" / "ab_routing").exists():
        raise RuntimeError(f"Unzipped repo does not contain cuda_ttl/ab_routing: {dest}")


def _find_repo_zip():
    """Find an already-present repo zip in the locations Colab commonly uses."""
    explicit = Path(LOCAL_REPO_ZIP).expanduser()
    search_roots = [
        explicit.parent,
        Path(WORK),
        Path.cwd(),
        Path("/content"),
        Path("/mnt/data"),  # works in ChatGPT sandbox, harmless in Colab if absent
    ]

    candidates = []
    if explicit.exists():
        candidates.append(explicit)
    for root in search_roots:
        try:
            if root.exists():
                candidates.extend(Path(x) for x in glob.glob(str(root / "rishabh-tlm*.zip")))
        except Exception:
            pass

    # de-dupe while preserving order
    out = []
    seen = set()
    for c in candidates:
        try:
            resolved = c.resolve()
        except Exception:
            continue
        if c.exists() and resolved not in seen:
            out.append(c)
            seen.add(resolved)
    return out[0] if out else None


def _upload_repo_zip_colab():
    """Upload a zip in Colab and return the actual path it was written to.

    Colab saves uploads into the current working directory, not necessarily /content.
    The previous notebook assumed /content/<name>, which caused FileNotFoundError.
    This version verifies the path and falls back to writing uploaded bytes manually.
    """
    from google.colab import files

    print("No local repo zip found. Upload rishabh-tlm-ab-routing-2026-07-02.zip now.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("No .zip file uploaded.")

    name = zip_names[0]
    candidates = [
        Path.cwd() / name,
        Path(WORK) / name,
        Path("/content") / name,
        Path(name),
    ]
    for candidate in candidates:
        if candidate.exists():
            print(f"Uploaded zip found at: {candidate.resolve()}")
            return candidate.resolve()

    # Defensive fallback: Colab returned bytes, so write them somewhere we control.
    fallback = Path(WORK) / name
    fallback.parent.mkdir(parents=True, exist_ok=True)
    fallback.write_bytes(uploaded[name])
    print(f"Uploaded zip bytes written to: {fallback.resolve()}")
    return fallback.resolve()


if repo_dir.exists():
    print("repo already exists:", repo_dir)
    # keep this non-destructive; uncomment if you explicitly want latest remote
    # subprocess.check_call(["git", "-C", str(repo_dir), "pull"])
else:
    clone_url = _clone_url_with_optional_token(REPO_URL)
    print("Cloning repo...")
    cp = _run_capture(["git", "clone", clone_url, str(repo_dir)])
    if cp.returncode != 0:
        print("git clone failed.")
        if cp.stdout.strip():
            print("--- git stdout ---")
            print(cp.stdout[-4000:])
        if cp.stderr.strip():
            print("--- git stderr ---")
            # Redact token if present.
            safe_stderr = cp.stderr
            token = os.environ.get(GITHUB_TOKEN_ENV, "").strip()
            if token:
                safe_stderr = safe_stderr.replace(token, "<GITHUB_TOKEN>")
            print(safe_stderr[-4000:])

        zip_path = _find_repo_zip()
        if zip_path is None:
            try:
                zip_path = _upload_repo_zip_colab()
            except Exception as upload_e:
                raise RuntimeError(
                    "Could not clone the repo and no repo zip fallback was available.\n\n"
                    "Most likely causes:\n"
                    "1) The GitHub repo is private and Colab has no auth. Set os.environ['GITHUB_TOKEN'] first.\n"
                    "2) The URL/branch is wrong or unavailable.\n"
                    "3) Colab/network blocked GitHub.\n\n"
                    f"Original git error:\n{cp.stderr[-2000:]}"
                ) from upload_e

        _unzip_repo(Path(zip_path), repo_dir)

os.chdir(repo_dir)
print("repo:", repo_dir)
try:
    head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
    print("git head:", head)
except Exception:
    print("git head unavailable: repo loaded from zip or non-git folder")

print("ab_routing files:")
for p in sorted((repo_dir / "cuda_ttl" / "ab_routing").glob("ab_*.py")):
    print(" -", p.relative_to(repo_dir))


In [ ]:
# Runtime patch: make model loading more robust for Qwen3.5 and add clear preflight checks.
# This is non-destructive to the repo ZIP; it only edits the working copy inside WORK.
from pathlib import Path
import subprocess, sys, os

if not Path("cuda_ttl/ab_routing/ab_build_pool.py").exists():
    raise FileNotFoundError("Run the repo setup cell first; cuda_ttl/ab_routing/ab_build_pool.py not found.")

# 1) ab_build_pool.py originally hard-coded AutoModelForCausalLM.
# For Qwen3.5, use the checkpoint-declared architecture when Transformers knows it;
# otherwise fall back to AutoModelForCausalLM.
p = Path("cuda_ttl/ab_routing/ab_build_pool.py")
s = p.read_text()
s = s.replace(
    '    from transformers import AutoModelForCausalLM, AutoTokenizer\n'
    '    from peft import LoraConfig, TaskType, get_peft_model\n'
    '    tokenizer = AutoTokenizer.from_pretrained(args.model)\n'
    '    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=torch.bfloat16,\n'
    '                                                 device_map="cuda")\n',
    '    import transformers\n'
    '    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer\n'
    '    from peft import LoraConfig, TaskType, get_peft_model\n'
    '    tokenizer = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)\n'
    '    cfg = AutoConfig.from_pretrained(args.model, trust_remote_code=True)\n'
    '    arch = (getattr(cfg, "architectures", None) or [""])[0]\n'
    '    model_cls = getattr(transformers, arch, None) or AutoModelForCausalLM\n'
    '    print(f"loading with {model_cls.__name__}")\n'
    '    model = model_cls.from_pretrained(args.model, torch_dtype=torch.bfloat16,\n'
    '                                      device_map="cuda", trust_remote_code=True)\n'
)
p.write_text(s)

# 2) Add trust_remote_code to train loader too, same idea.
p = Path("cuda_ttl/ab_routing/ab_train_ttl.py")
s = p.read_text()
s = s.replace('    tokenizer = AutoTokenizer.from_pretrained(args.model)\n', '    tokenizer = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)\n')
s = s.replace('    cfg = AutoConfig.from_pretrained(args.model)\n', '    cfg = AutoConfig.from_pretrained(args.model, trust_remote_code=True)\n')
s = s.replace('    model = model_cls.from_pretrained(args.model, torch_dtype=torch.bfloat16,\n                                      device_map="cuda")\n',
              '    model = model_cls.from_pretrained(args.model, torch_dtype=torch.bfloat16,\n                                      device_map="cuda", trust_remote_code=True)\n')
p.write_text(s)

# 3) Hard preflight before burning time.
import torch, transformers
from transformers import AutoConfig, AutoTokenizer
print("MODEL =", MODEL)
print("torch =", torch.__version__)
print("transformers =", transformers.__version__)
print("cuda =", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab: Runtime -> Change runtime type -> GPU.")
print("gpu =", torch.cuda.get_device_name(0))
try:
    tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    cfg = AutoConfig.from_pretrained(MODEL, trust_remote_code=True)
    print("model_type =", getattr(cfg, "model_type", None))
    print("architectures =", getattr(cfg, "architectures", None))
    print("tokenizer ok; chat_template =", bool(getattr(tok, "chat_template", None)))
except Exception as e:
    raise RuntimeError(
        "Model/tokenizer preflight failed. If MODEL is Qwen/Qwen3.5-0.8B, rerun the install cell, "
        "restart runtime if needed, or set MODEL='Qwen/Qwen2.5-0.5B-Instruct' for a text-only smoke test."
    ) from e

print("patched scripts ok")
for file in ["ab_build_pool.py", "ab_train_ttl.py"]:
    subprocess.check_call([sys.executable, "-m", "py_compile", str(Path("cuda_ttl/ab_routing") / file)])
    print("py_compile ok:", file)


In [ ]:
# Patch generation to avoid vLLM CUDA-wheel failures on Colab.
# The repo's ab_generate.py imports vLLM unconditionally. If your vLLM wheel expects
# CUDA 13 but Colab only has CUDA 12 libraries, it crashes on import before inference.
# This replacement keeps the same CLI/output format but uses Transformers greedy generation.
from pathlib import Path
import subprocess, sys

gen_path = Path("cuda_ttl/ab_routing/ab_generate.py")
if not gen_path.exists():
    raise FileNotFoundError("Run repo setup first; ab_generate.py not found")

gen_path.write_text('#!/usr/bin/env python\n"""\nStage 3: greedy predictions for an AdaptEval bench.\n\nPatched notebook version: uses Hugging Face Transformers instead of vLLM so Colab\nis not blocked by vLLM/CUDA wheel mismatches such as missing libcudart.so.13.\nOutput rows {idx, question, label, predict} match the original script.\n"""\n\nimport argparse\nimport json\nimport os\nimport time\n\n\ndef build_user(rec):\n    if "instruction" in rec:\n        instruction = (rec.get("instruction") or "").strip()\n        extra = (rec.get("input") or "").strip()\n        return instruction + (("\n" + extra) if extra else "")\n    return (rec.get("question") or "").strip()\n\n\ndef gold_label(rec):\n    if rec.get("answer") not in (None, ""):\n        return str(rec["answer"]).strip()\n    val = rec.get("output") if "output" in rec else rec.get("answers")\n    return str(val or "").strip()\n\n\ndef _load_model_and_tokenizer(model_name_or_path):\n    import torch\n    import transformers\n    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer\n\n    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "left"\n\n    cfg = AutoConfig.from_pretrained(model_name_or_path, trust_remote_code=True)\n    arch = (getattr(cfg, "architectures", None) or [""])[0]\n    model_cls = getattr(transformers, arch, None) or AutoModelForCausalLM\n    print(f"loading with {model_cls.__name__}")\n\n    model = model_cls.from_pretrained(\n        model_name_or_path,\n        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,\n        device_map="auto" if torch.cuda.is_available() else None,\n        trust_remote_code=True,\n    )\n    model.eval()\n    return model, tokenizer\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="Transformers greedy predictions for AdaptEval")\n    ap.add_argument("--model", required=True)\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--max-samples", type=int, default=1000)\n    ap.add_argument("--max-new-tokens", type=int, default=1024)\n    ap.add_argument("--gpu-mem", type=float, default=0.85, help="kept for CLI compatibility; ignored")\n    ap.add_argument("--max-model-len", type=int, default=4096)\n    ap.add_argument("--batch-size", type=int, default=8)\n    args = ap.parse_args()\n\n    records = json.load(open(args.data, encoding="utf-8"))[: args.max_samples]\n\n    model, tokenizer = _load_model_and_tokenizer(args.model)\n\n    def chat(user):\n        msgs = [{"role": "user", "content": user}]\n        try:\n            return tokenizer.apply_chat_template(\n                msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False\n            )\n        except TypeError:\n            return tokenizer.apply_chat_template(\n                msgs, tokenize=False, add_generation_prompt=True\n            )\n\n    built = []\n    for idx, rec in enumerate(records):\n        user = build_user(rec)\n        if not user:\n            continue\n        built.append((idx, user, gold_label(rec), chat(user)))\n\n    import torch\n    t0 = time.perf_counter()\n    predictions = []\n\n    for start in range(0, len(built), args.batch_size):\n        chunk = built[start : start + args.batch_size]\n        prompts = [b[3] for b in chunk]\n        enc = tokenizer(\n            prompts,\n            return_tensors="pt",\n            padding=True,\n            truncation=True,\n            max_length=args.max_model_len,\n        )\n        enc = {k: v.to(model.device) for k, v in enc.items()}\n        input_width = enc["input_ids"].shape[1]\n\n        with torch.inference_mode():\n            out_ids = model.generate(\n                **enc,\n                do_sample=False,\n                max_new_tokens=args.max_new_tokens,\n                pad_token_id=tokenizer.pad_token_id,\n                eos_token_id=tokenizer.eos_token_id,\n            )\n\n        gen_ids = out_ids[:, input_width:]\n        texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)\n        predictions.extend(texts)\n\n        done = min(start + len(chunk), len(built))\n        if done % 100 == 0 or done == len(built):\n            print(f"generated {done}/{len(built)}", flush=True)\n\n    print(f"{len(built)} prompts in {time.perf_counter()-t0:.1f}s")\n\n    os.makedirs(os.path.dirname(args.output) or ".", exist_ok=True)\n    with open(args.output, "w", encoding="utf-8") as f:\n        for (idx, user, label, _p), pred in zip(built, predictions):\n            f.write(json.dumps({"idx": idx, "question": user, "label": label,\n                                "predict": pred}, ensure_ascii=False) + "\n")\n    print(f"-> {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n')
subprocess.check_call([sys.executable, "-m", "py_compile", str(gen_path)])
print("patched and py_compile ok:", gen_path)


In [ ]:
# Data paths expected by cuda_ttl/ab_routing/run_ab.sh, but we call scripts directly.
DATA_FILES = {
    "logiqa": "data/AdaptEval/logiqa_random_5k.json",
    "medicine": "data/AdaptEval/medicine_mcqa_random_5k.json",
    "geography": "data/AdaptEval/geography_mmlu.json",
    "gsm8k": "data/AdaptEval/gsm8k_random_5k.json",
}

missing = [p for ds, p in DATA_FILES.items() if ds in DATASETS and not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Missing dataset files: " + ", ".join(missing) +
        "\nThe uploaded repo snapshot included these under data/AdaptEval/. If your clone does not, copy them there before running."
    )

for ds in DATASETS:
    print(ds, "->", DATA_FILES[ds], "exists", Path(DATA_FILES[ds]).exists())

## Build artifacts (pool -> train -> base gen -> adapted gen)

This runs the same first four stages as before and writes, per dataset, under
`WORK/saves_dynamic_v2ce/<ds>/`: `pool/features.jsonl`, `preds_base.jsonl`,
`preds_adapted.jsonl`, and `timing_summary.json`. The routing evaluation itself is done
in-notebook further down, so the `ab_route_eval.py` / `ab_dynamic_ce_from_v2.py` stages are
dropped. With `SKIP_IF_EXISTS=True`, existing outputs are reused and their cached timing is
carried into the cost model.

In [ ]:
import subprocess, os, json, time, shlex, sys
from pathlib import Path

SCRIPTS = Path("cuda_ttl/ab_routing").resolve()
ARTIFACT_ROOT = Path(WORK) / "saves_dynamic_v2ce"
LOG_ROOT = Path(WORK) / "logs_dynamic_v2ce"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

# Only the artifact-producing stages remain; routing is evaluated in-notebook.
STAGE_ORDER = ["pool", "train", "gen_base", "gen_adapted"]


def _read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return default


def run_cmd(cmd, log_path=None, skip_if=None, cached_stage=None, timing_cache=None):
    """Run a command, returning a timing record. Reuses cached elapsed time when skipping."""
    skip_path = Path(skip_if) if skip_if is not None else None
    if skip_path is not None and SKIP_IF_EXISTS and skip_path.exists():
        cached = (timing_cache or {}).get("stages", {}).get(cached_stage or "")
        elapsed = cached.get("elapsed_seconds") if isinstance(cached, dict) else None
        print(f"SKIP exists: {skip_path}" + (f" | cached {elapsed/60:.1f} min" if elapsed else " | no cached timing"))
        return {"stage": cached_stage, "elapsed_seconds": elapsed, "skipped": True, "returncode": 0,
                "cmd": [str(c) for c in cmd]}

    print("RUN:", " ".join(shlex.quote(str(c)) for c in cmd))
    t0 = time.perf_counter()
    if log_path:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, "w", encoding="utf-8") as log:
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:
                print(line, end="")
                log.write(line)
            rc = p.wait()
    else:
        rc = subprocess.call(cmd)
    elapsed = time.perf_counter() - t0
    print(f"rc={rc} elapsed={elapsed/60:.1f} min")
    if rc != 0:
        if log_path and Path(log_path).exists():
            print("\n----- LAST 160 LOG LINES -----")
            print("\n".join(Path(log_path).read_text(errors="replace").splitlines()[-160:]))
            print("----- END LOG TAIL -----\n")
        raise RuntimeError(f"Command failed with rc={rc}: {cmd}\nLog: {log_path}")
    return {"stage": cached_stage, "elapsed_seconds": elapsed, "skipped": False, "returncode": rc,
            "cmd": [str(c) for c in cmd]}


def write_timing_summary(ds, d, stage_records):
    timing_path = d / "timing_summary.json"
    vals = [stage_records.get(n, {}).get("elapsed_seconds") for n in STAGE_ORDER]
    full = None if any(v is None for v in vals) else float(sum(vals))
    measured = sum(r.get("elapsed_seconds") or 0.0 for r in stage_records.values() if not r.get("skipped"))
    summary = {"dataset": ds,
               "timing_note": "wall-clock seconds; skipped stages reuse cached elapsed when available",
               "full_pipeline_seconds": full, "measured_this_notebook_seconds": measured,
               "stages": stage_records}
    with open(timing_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print("wrote", timing_path)
    return timing_path


def build_artifacts(ds):
    n = N_BY_DATASET[ds]
    data = DATA_FILES[ds]
    d = ARTIFACT_ROOT / ds
    pool = d / "pool"
    adapter = d / "adapter"
    base_preds = d / "preds_base.jsonl"
    adapted_preds = d / "preds_adapted.jsonl"
    timing_path = d / "timing_summary.json"
    d.mkdir(parents=True, exist_ok=True)

    old_timing = _read_json(timing_path, default={}) or {}
    stage_records = dict(old_timing.get("stages", {}))

    def record(stage, cmd, log_path, skip_if):
        rec = run_cmd(cmd, log_path, skip_if=skip_if, cached_stage=stage, timing_cache=old_timing)
        stage_records[stage] = rec
        write_timing_summary(ds, d, stage_records)
        return rec

    record("pool",
           [sys.executable, str(SCRIPTS / "ab_build_pool.py"), "--model", MODEL, "--data", data,
            "--dataset", ds, "--max-samples", str(n), "--out-dir", str(pool)],
           LOG_ROOT / f"{ds}_pool.log", pool / "features.jsonl")
    record("train",
           [sys.executable, str(SCRIPTS / "ab_train_ttl.py"), "--model", MODEL, "--data", data,
            "--selection-file", str(pool / "selection.json"), "--max-samples", str(n),
            "--output-dir", str(adapter)],
           LOG_ROOT / f"{ds}_train.log", adapter / "merged" / "config.json")
    record("gen_base",
           [sys.executable, str(SCRIPTS / "ab_generate.py"), "--model", MODEL, "--data", data,
            "--max-samples", str(n), "--output", str(base_preds)],
           LOG_ROOT / f"{ds}_gen_base.log", base_preds)
    record("gen_adapted",
           [sys.executable, str(SCRIPTS / "ab_generate.py"), "--model", str(adapter / "merged"),
            "--data", data, "--max-samples", str(n), "--output", str(adapted_preds)],
           LOG_ROOT / f"{ds}_gen_adapted.log", adapted_preds)

    write_timing_summary(ds, d, stage_records)
    return d


if RUN_FULL_PIPELINE:
    artifact_dirs = [build_artifacts(ds) for ds in DATASETS]
else:
    artifact_dirs = [ARTIFACT_ROOT / ds for ds in DATASETS]
    print("RUN_FULL_PIPELINE=False; using existing artifacts under", ARTIFACT_ROOT)


## Frontier experiment configuration

Each signal is tagged with its cost **regime**:

- `pregen` (`q_ce`, `admission_score`): available from the prompt prefill, so deployment runs
  exactly one model per item. Cost interpolates between base-only and adapted-only.
- `postgen` (`out_ce`): requires a full base generation first, so base runs on everything and
  adapted is added for the routed fraction.

`SIGNAL_FWD_COST_FRAC` is the marginal cost of computing a pre-gen signal, as a fraction of one
base generation. `q_ce`/`admission_score` fall out of the prefill you already pay for before
decoding, so `0.0` is the honest default; bump it if you want a pessimistic sensitivity check.

In [ ]:
import numpy as np, random, math

SIGNALS = ["q_ce", "admission_score", "out_ce"]     # candidate routing signals
SIGNAL_REGIME = {"q_ce": "pregen", "admission_score": "pregen", "out_ce": "postgen"}

FIXED_CE_TAU = 0.3          # the old fixed baseline, for reference
VAL_FRAC = 0.5             # fraction of each dataset used to pick the operating point
SPLIT_SEED = 0
FRACTION_GRID = [round(x, 4) for x in np.linspace(0.0, 1.0, 51)]  # routed-fraction sweep
SIGNAL_FWD_COST_FRAC = 0.0  # marginal cost of a pre-gen signal, as a fraction of one base gen

ARTIFACT_ROOT = Path(WORK) / "saves_dynamic_v2ce"
print("signals:", SIGNALS, "| regimes:", SIGNAL_REGIME)


## Core: load artifacts, grade both arms, sweep signals, cost each regime

Pure post-hoc analysis over `features.jsonl` + `preds_base.jsonl` + `preds_adapted.jsonl` + `timing_summary.json`. No regeneration.

In [ ]:
import sys, json
from pathlib import Path

# grading.py lives in the repo; cwd is the repo root after the clone cell.
_grading_dir = str(Path("cuda_ttl/ab_routing").resolve())
if _grading_dir not in sys.path:
    sys.path.insert(0, _grading_dir)
import grading  # noqa: E402


def _load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def _grade(pred_path, answer_type):
    ok = {}
    text_len = {}
    for row in _load_jsonl(pred_path):
        i = int(row["idx"])
        pred = str(row.get("predict", ""))
        ex = grading.extract(pred, answer_type)
        ok[i] = bool(grading.is_correct(ex.value, str(row["label"]), answer_type))
        text_len[i] = pred            # keep raw text for the token-length diagnostic
    return ok, text_len


def load_dataset(ds, artifact_root=ARTIFACT_ROOT, signals=SIGNALS):
    d = Path(artifact_root) / ds
    feats_path = next((c for c in [d / "pool" / "features.jsonl", d / "features.jsonl"] if c.exists()), None)
    base_path, adapted_path = d / "preds_base.jsonl", d / "preds_adapted.jsonl"
    timing_path = d / "timing_summary.json"
    for label, path in [("features.jsonl", feats_path), ("preds_base.jsonl", base_path),
                        ("preds_adapted.jsonl", adapted_path)]:
        if path is None or not Path(path).exists():
            raise FileNotFoundError(f"[{ds}] missing {label} under {d}. Run the pipeline cell first.")

    answer_type = grading.detect_answer_type(ds)
    if answer_type is None:
        raise SystemExit(f"[{ds}] not a deterministic/verifiable bench known to grading.py")

    base_ok, base_txt = _grade(base_path, answer_type)
    adapted_ok, adapted_txt = _grade(adapted_path, answer_type)
    feats = {int(r["idx"]): r for r in _load_jsonl(feats_path)}

    present = [s for s in signals if any(feats[i].get(s) is not None for i in feats)]
    dropped_signals = [s for s in signals if s not in present]

    idx = sorted(i for i in base_ok if i in adapted_ok and i in feats
                 and all(feats[i].get(s) is not None for s in present))
    if not idx:
        raise SystemExit(f"[{ds}] no overlapping rows across base/adapted/features with all signals")

    sig = {s: {i: float(feats[i][s]) for i in idx} for s in present}
    route_choice = {i: feats[i].get("route_choice") for i in idx}
    v2_set = {i for i in idx if route_choice.get(i) == "adapted"}
    timing = json.load(open(timing_path)) if timing_path.exists() else {}

    return {"ds": ds, "idx": idx, "base_ok": base_ok, "adapted_ok": adapted_ok,
            "sig": sig, "present_signals": present, "dropped_signals": dropped_signals,
            "v2_set": v2_set, "base_txt": base_txt, "adapted_txt": adapted_txt,
            "timing": timing, "answer_type": answer_type,
            "n": len(idx), "n_all_pred": len(base_ok)}


# ---- split, routing, sweep ----
def split_val_test(idx, seed=SPLIT_SEED, val_frac=VAL_FRAC):
    order = list(idx)
    random.Random(seed).shuffle(order)
    k = int(round(len(order) * val_frac))
    return set(order[:k]), set(order[k:])


def route_acc(subset, base_ok, adapted_ok, adapted_set):
    subset = list(subset)
    if not subset:
        return float("nan")
    return sum((adapted_ok[i] if i in adapted_set else base_ok[i]) for i in subset) / len(subset)


def top_by(subset, sig_map, frac, higher_is_adapted):
    subset = list(subset)
    n = len(subset)
    k = int(round(n * max(0.0, min(1.0, frac))))
    order = sorted(subset, key=lambda i: sig_map[i], reverse=higher_is_adapted)
    return set(order[:k]), (k / max(1, n))


def sweep_curve(subset, sig_map, base_ok, adapted_ok, grid, higher_is_adapted):
    out = []
    for f in grid:
        sel, af = top_by(subset, sig_map, f, higher_is_adapted)
        out.append({"target_frac": f, "adapted_frac": af,
                    "acc": route_acc(subset, base_ok, adapted_ok, sel)})
    return out


def choose_operating_point(val, sig_map, base_ok, adapted_ok, grid):
    """Pick (direction, target_frac) that maximizes VAL accuracy; tie-break to the cheaper (smaller) frac."""
    best_key, best = None, None
    for direction in (True, False):
        for f in grid:
            sel, af = top_by(val, sig_map, f, direction)
            acc = route_acc(val, base_ok, adapted_ok, sel)
            key = (acc, -af)
            if best_key is None or key > best_key:
                best_key, best = key, (direction, f)
    return best  # (higher_is_adapted, target_frac)


# ---- cost model (per item, seconds) ----
def per_item_costs(data):
    stages = (data.get("timing") or {}).get("stages", {})
    def sec(name):
        v = stages.get(name, {})
        return v.get("elapsed_seconds") if isinstance(v, dict) else None
    n = data["n_all_pred"] or data["n"]
    gb, ga = sec("gen_base"), sec("gen_adapted")
    c_base = (gb / n) if (gb and n) else None
    c_adapted = (ga / n) if (ga and n) else None
    return c_base, c_adapted


def cost_at(frac, regime, c_base, c_adapted, signal_cost):
    if c_base is None or c_adapted is None:
        return None
    if regime == "postgen":          # base on all + adapted on routed (out_ce)
        return c_base + frac * c_adapted
    return (1.0 - frac) * c_base + frac * c_adapted + signal_cost  # pregen interpolation

print("core loaded")


## Run the frontier analysis

For every dataset we compute, on the held-out **test** split:

- the four bounds (base-only, adapted-only, oracle ceiling, V2) plus fixed `CE@0.3`;
- for each signal, the val-frozen operating point evaluated on test (accuracy, routed fraction, est cost, winning direction);
- the full test frontier curve for plotting.

Operating points are chosen on **validation only**, so the reported test accuracy is leakage-free.

In [ ]:
def analyze(data):
    idx = data["idx"]
    base_ok, adapted_ok = data["base_ok"], data["adapted_ok"]
    val, test = split_val_test(idx)
    c_base, c_adapted = per_item_costs(data)
    signal_cost = (SIGNAL_FWD_COST_FRAC * c_base) if c_base is not None else 0.0

    # bounds on the TEST split
    base_only = route_acc(test, base_ok, adapted_ok, set())
    adapted_only = route_acc(test, base_ok, adapted_ok, set(test))
    oracle = sum((base_ok[i] or adapted_ok[i]) for i in test) / len(test)
    v2_test = data["v2_set"] & test
    v2_frac = len(v2_test) / len(test)
    v2_acc = route_acc(test, base_ok, adapted_ok, v2_test)

    fixed_set = {i for i in test if data["sig"].get("out_ce", {}).get(i, -1e9) >= FIXED_CE_TAU} \
        if "out_ce" in data["sig"] else set()
    fixed_frac = (len(fixed_set) / len(test)) if "out_ce" in data["sig"] else None
    fixed_acc = route_acc(test, base_ok, adapted_ok, fixed_set) if "out_ce" in data["sig"] else None

    bounds = {
        "dataset": data["ds"], "n_test": len(test), "n_val": len(val),
        "base_only_acc": base_only, "adapted_only_acc": adapted_only, "oracle_acc": oracle,
        "v2_acc": v2_acc, "v2_frac": v2_frac,
        "fixed_ce_acc": fixed_acc, "fixed_ce_frac": fixed_frac,
        "c_base_s": c_base, "c_adapted_s": c_adapted,
        "adapted_cheaper": (None if (c_base is None or c_adapted is None) else c_adapted < c_base),
    }

    signal_rows, curves = [], {}
    for s in data["present_signals"]:
        sig_map = data["sig"][s]
        regime = SIGNAL_REGIME.get(s, "pregen")
        direction, f_star = choose_operating_point(val, sig_map, base_ok, adapted_ok, FRACTION_GRID)
        sel, af = top_by(test, sig_map, f_star, direction)
        acc = route_acc(test, base_ok, adapted_ok, sel)
        cost = cost_at(af, regime, c_base, c_adapted, signal_cost)
        signal_rows.append({
            "dataset": data["ds"], "signal": s, "regime": regime,
            "direction": "high->adapted" if direction else "low->adapted",
            "test_acc": acc, "test_adapted_frac": af, "est_cost_s_per_item": cost,
            "val_frac_target": f_star,
        })
        curves[s] = {
            "regime": regime, "direction": direction,
            "points": [
                {"adapted_frac": p["adapted_frac"], "acc": p["acc"],
                 "cost": cost_at(p["adapted_frac"], regime, c_base, c_adapted, signal_cost)}
                for p in sweep_curve(test, sig_map, base_ok, adapted_ok, FRACTION_GRID, direction)
            ],
        }

    return {"bounds": bounds, "signals": signal_rows, "curves": curves,
            "c_base": c_base, "c_adapted": c_adapted, "signal_cost": signal_cost}


results = {}
for ds in DATASETS:
    try:
        data = load_dataset(ds)
        results[ds] = analyze(data)
        if data["dropped_signals"]:
            print(f"[{ds}] note: signals absent in features.jsonl -> {data['dropped_signals']}")
    except (FileNotFoundError, SystemExit) as e:
        print(f"[{ds}] SKIPPED: {e}")

print("analyzed:", list(results))


## Bounds table (the numbers the old table omitted)

base-only vs adapted-only vs **oracle ceiling** vs V2 vs fixed `CE@0.3`, on held-out test. Also whether adapted generation is cheaper than base for that dataset.

In [ ]:
import pandas as pd

def pct(x):
    return "" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{100*x:.1f}%"

def secs(x):
    return "" if x is None else f"{x:.4f}"

brows = []
for ds, r in results.items():
    b = r["bounds"]
    brows.append({
        "dataset": ds, "n_test": b["n_test"],
        "base-only": pct(b["base_only_acc"]), "adapted-only": pct(b["adapted_only_acc"]),
        "oracle": pct(b["oracle_acc"]), "V2": pct(b["v2_acc"]), "V2 frac": pct(b["v2_frac"]),
        "fixed CE@0.3": pct(b["fixed_ce_acc"]), "fixed frac": pct(b["fixed_ce_frac"]),
        "base gen s/item": secs(b["c_base_s"]), "adapted gen s/item": secs(b["c_adapted_s"]),
        "adapted cheaper?": ("" if b["adapted_cheaper"] is None else ("YES" if b["adapted_cheaper"] else "no")),
    })
bounds_df = pd.DataFrame(brows)
try:
    display(bounds_df)
except NameError:
    print(bounds_df.to_string(index=False))


## Per-signal frozen operating points (val-chosen, test-reported)

Each signal's threshold is picked on validation and evaluated on test. `est cost` is per item in seconds under that signal's regime; the last two columns compare it to the two single-model baselines.

In [ ]:
srows = []
for ds, r in results.items():
    c_base, c_adapted = r["c_base"], r["c_adapted"]
    for row in r["signals"]:
        cost = row["est_cost_s_per_item"]
        d_base = None if (cost is None or c_base is None) else (cost - c_base) / c_base
        d_adp = None if (cost is None or c_adapted is None) else (cost - c_adapted) / c_adapted
        srows.append({
            "dataset": ds, "signal": row["signal"], "regime": row["regime"],
            "direction": row["direction"], "test acc": pct(row["test_acc"]),
            "adapted frac": pct(row["test_adapted_frac"]), "est cost s/item": secs(cost),
            "vs base-only cost": ("" if d_base is None else f"{100*d_base:+.1f}%"),
            "vs adapted-only cost": ("" if d_adp is None else f"{100*d_adp:+.1f}%"),
        })
signals_df = pd.DataFrame(srows)
try:
    display(signals_df)
except NameError:
    print(signals_df.to_string(index=False))


## Cost-quality frontier (the main figure)

x = estimated cost per item (s), y = test accuracy. One curve per signal (costed under its regime), with base-only / adapted-only / V2 / fixed-CE points and the oracle ceiling. A signal is interesting only if its curve pushes up-and-left of the base-only point toward oracle.

In [ ]:
import matplotlib.pyplot as plt

for ds, r in results.items():
    b = r["bounds"]
    c_base, c_adapted, sc = r["c_base"], r["c_adapted"], r["signal_cost"]
    if c_base is None or c_adapted is None:
        print(f"[{ds}] no generation timing on disk; skipping cost-quality plot (see acc-vs-fraction below).")
        continue

    plt.figure(figsize=(7, 5))
    for s, cur in r["curves"].items():
        xs = [p["cost"] for p in cur["points"]]
        ys = [100 * p["acc"] for p in cur["points"]]
        plt.plot(xs, ys, marker=".", linewidth=1, label=f"{s} ({cur['regime']})")

    plt.scatter([c_base], [100 * b["base_only_acc"]], s=90, marker="s", zorder=5, label="base-only")
    plt.scatter([c_adapted], [100 * b["adapted_only_acc"]], s=90, marker="D", zorder=5, label="adapted-only")
    v2_cost = cost_at(b["v2_frac"], "pregen", c_base, c_adapted, sc)
    plt.scatter([v2_cost], [100 * b["v2_acc"]], s=90, marker="^", zorder=5, label="V2")
    if b["fixed_ce_acc"] is not None:
        fx_cost = cost_at(b["fixed_ce_frac"], "postgen", c_base, c_adapted, sc)
        plt.scatter([fx_cost], [100 * b["fixed_ce_acc"]], s=90, marker="v", zorder=5, label="fixed CE@0.3")
    plt.axhline(100 * b["oracle_acc"], color="grey", linestyle="--", linewidth=1, label="oracle ceiling")

    plt.xlabel("est. cost per item (s)")
    plt.ylabel("test accuracy (%)")
    plt.title(f"{ds}: cost-quality frontier by routing signal")
    plt.legend(fontsize=8, loc="best")
    plt.grid(True, alpha=0.3)
    plt.show()


## Accuracy vs routed fraction (flat-optimum / direction check)

If a curve is flat near its peak, the old notebook's "best multiplier = 2.5x" was an artifact of a flat optimum and the V2-fraction anchor was not localizing anything.

In [ ]:
for ds, r in results.items():
    plt.figure(figsize=(7, 5))
    for s, cur in r["curves"].items():
        xs = [p["adapted_frac"] for p in cur["points"]]
        ys = [100 * p["acc"] for p in cur["points"]]
        plt.plot(xs, ys, marker=".", linewidth=1, label=f"{s} ({cur['direction'] and 'high->adapted' or 'low->adapted'})")
    b = r["bounds"]
    plt.axhline(100 * b["base_only_acc"], color="tab:blue", linestyle=":", linewidth=1, label="base-only")
    plt.axhline(100 * b["adapted_only_acc"], color="tab:orange", linestyle=":", linewidth=1, label="adapted-only")
    plt.axhline(100 * b["oracle_acc"], color="grey", linestyle="--", linewidth=1, label="oracle")
    plt.xlabel("routed-to-adapted fraction")
    plt.ylabel("test accuracy (%)")
    plt.title(f"{ds}: accuracy vs routed fraction")
    plt.legend(fontsize=8, loc="best")
    plt.grid(True, alpha=0.3)
    plt.show()


## Diagnostic: why does base generation dominate cost?

Hypothesis: on medicine the base model runs to `max_new_tokens` (rambles / never emits the
answer format) while the LoRA-adapted model answers concisely, making adapted generation far
cheaper. This measures mean generated length (base vs adapted) using the model tokenizer when
available, falling back to whitespace tokens. A large base/adapted ratio explains the cost
inversion and tells you which direction any cascade should run.

In [ ]:
def _make_counter():
    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        return (lambda t: len(tok.encode(t or "", add_special_tokens=False))), "tokenizer"
    except Exception as e:
        print("tokenizer unavailable, using whitespace word count:", repr(e))
        return (lambda t: len((t or "").split())), "words"

_count, _mode = _make_counter()

drows = []
for ds in DATASETS:
    if ds not in results:
        continue
    try:
        data = load_dataset(ds)
    except (FileNotFoundError, SystemExit):
        continue
    idx = data["idx"]
    bl = [_count(data["base_txt"].get(i, "")) for i in idx]
    al = [_count(data["adapted_txt"].get(i, "")) for i in idx]
    mb = sum(bl) / len(bl) if bl else float("nan")
    ma = sum(al) / len(al) if al else float("nan")
    drows.append({
        "dataset": ds, f"base mean len ({_mode})": f"{mb:.1f}",
        f"adapted mean len ({_mode})": f"{ma:.1f}",
        "base/adapted ratio": f"{(mb/ma):.1f}x" if ma else "",
        "base gen s/item": secs(results[ds]["bounds"]["c_base_s"]),
        "adapted gen s/item": secs(results[ds]["bounds"]["c_adapted_s"]),
    })
tok_df = pd.DataFrame(drows)
try:
    display(tok_df)
except NameError:
    print(tok_df.to_string(index=False))


## Verdict per dataset

Mechanical read of the tables above: the best deployable accuracy, whether a **pre-gen** router beats base-only on cost at >= base accuracy (the clean win the old notebook wanted), and whether one arm just dominates outright.

In [ ]:
def verdict(ds, r):
    b = r["bounds"]
    c_base, c_adapted = r["c_base"], r["c_adapted"]
    lines = [f"### {ds}"]

    # deployable options: (name, acc, cost, is_pregen)
    opts = [("base-only", b["base_only_acc"], c_base, True),
            ("adapted-only", b["adapted_only_acc"], c_adapted, True)]
    for row in r["signals"]:
        opts.append((f"route:{row['signal']}", row["test_acc"], row["est_cost_s_per_item"],
                     row["regime"] == "pregen"))

    best_acc = max(opts, key=lambda o: (o[1] if o[1] is not None else -1))
    lines.append(f"- Best test accuracy: **{best_acc[0]}** at {100*best_acc[1]:.1f}% "
                 f"(oracle ceiling {100*b['oracle_acc']:.1f}%).")

    if b["adapted_cheaper"]:
        lines.append(f"- Adapted generation is **cheaper** than base here "
                     f"({b['c_adapted_s']:.3f}s vs {b['c_base_s']:.3f}s per item), so the cheap arm is adapted.")
        if b["adapted_only_acc"] >= b["base_only_acc"]:
            lines.append("- adapted-only is both **more accurate and cheaper** than base-only -> "
                         "it dominates; an out_ce router (base-on-all) is strictly wasteful here.")
    elif b["adapted_cheaper"] is False:
        lines.append(f"- Base generation is cheaper ({b['c_base_s']:.3f}s vs {b['c_adapted_s']:.3f}s), "
                     "so a cascade should keep cheap base as default and escalate to adapted.")

    # clean-win test: a pregen router cheaper than base-only at >= base accuracy
    clean = []
    if c_base is not None:
        for row in r["signals"]:
            if row["regime"] != "pregen" or row["est_cost_s_per_item"] is None:
                continue
            if row["est_cost_s_per_item"] < c_base and row["test_acc"] >= b["base_only_acc"] - 1e-9:
                clean.append((row["signal"], row["test_acc"], row["est_cost_s_per_item"]))
    if clean:
        for s, a, c in clean:
            lines.append(f"- CLEAN WIN: `{s}` routes at {100*a:.1f}% (>= base) for {c:.3f}s/item "
                         f"(< base {c_base:.3f}s). Cheaper than base-only at no accuracy loss.")
    else:
        lines.append("- No pre-gen router is strictly cheaper than base-only at >= base accuracy "
                     "on this split (check whether one arm already dominates).")

    # is out_ce carrying its cost premium?
    outce = next((row for row in r["signals"] if row["signal"] == "out_ce"), None)
    if outce and c_base is not None and outce["est_cost_s_per_item"] is not None:
        oc_cost = outce["est_cost_s_per_item"]
        oc_acc = 100 * outce["test_acc"]
        lines.append(f"- `out_ce` costs {oc_cost:.3f}s/item (>= base {c_base:.3f}s by construction, "
                     f"since it needs base on every item) for {oc_acc:.1f}% acc.")
    return "\n".join(lines)


from IPython.display import Markdown, display as _disp
for ds, r in results.items():
    try:
        _disp(Markdown(verdict(ds, r)))
    except Exception:
        print(verdict(ds, r))


In [ ]:
# Persist tables for the writeup.
out_dir = Path(WORK) / "frontier_out"
out_dir.mkdir(parents=True, exist_ok=True)
bounds_df.to_csv(out_dir / "bounds.csv", index=False)
signals_df.to_csv(out_dir / "signal_operating_points.csv", index=False)
tok_df.to_csv(out_dir / "token_length_diagnostic.csv", index=False)
with open(out_dir / "frontier_curves.json", "w", encoding="utf-8") as f:
    json.dump({ds: r["curves"] for ds, r in results.items()}, f, indent=2)
print("wrote:", *[p.name for p in out_dir.iterdir()])


## What to report

- Lead with the **bounds table**. If adapted-only already beats base-only at lower cost (the medicine pattern), the honest finding is "serve adapted," and any base-on-all router is wasteful there. The oracle gap tells you how much a *perfect* router could add over the better single arm; a small gap means routing effort is better spent improving the adapted model.
- Report the **cost-quality frontier** as the main figure. `out_ce` curves start at the base-only cost and can only move right (more expensive) - that is the structural reason the old notebook never found a cheaper-than-fixed win. `q_ce` / `admission_score` curves interpolate between the two single-model points, so they are the only candidates that can land below base-only cost.
- Every operating point here is **val-chosen, test-reported**, which removes the leakage the old notebook flagged (best multiplier tuned on the reported split).
- The base-vs-adapted **cost ratio is not constant across datasets** (near 1x for geography/logiqa, large for medicine per the token diagnostic), so the right cascade direction is dataset-dependent. State the ratio alongside each result.
- If you want a single deployable rule: pick the signal whose test frontier is closest to the oracle ceiling at or below base-only cost, freeze its val threshold, and report the frozen test point.
